# krat · prueba del avatar en una T4

Mide si una T4 genera **en tiempo real** la cara del avatar (Ditto) y la voz clonada (Seed-VC). Es una prueba desechable: decide si se construye la sección «Añadir avatar» con esta arquitectura.

1. **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU.**
2. **Entorno de ejecución → Ejecutar todo.** Te pedirá acceso a Drive y que subas 3 archivos. El nombre tiene que empezar como se indica; la extensión da igual:
   - `foto…` (jpg o png): tu cara de frente, bien iluminada, mirando a la cámara.
   - `mi_voz…` (wav, m4a, webm…): de 20 a 30 s leyendo en voz alta, sin ruido de fondo.
   - `gemini…`: de 10 a 15 s de una respuesta de Gemini (por ejemplo, grabada de una llamada con krat).

   Si cancelas la subida, se usan los ejemplos de los repositorios. Sirven para medir la velocidad, no para juzgar el parecido.
3. La primera vez, instalar y descargar tarda unos 15 min. Los pesos quedan en Drive (`krat_avatar/`). La conversión a TensorRT tarda de 30 a 40 min más y también se hace una sola vez.
4. Al final verás una tabla, un veredicto, el audio convertido y el vídeo `krat_avatar/salidas/ditto_resultado.mp4`. **Copia la tabla y pégasela a Claude.**

In [ ]:
# Paso 1 · Entorno
import glob, json, os, subprocess, sys, time

assert subprocess.run(["nvidia-smi"]).returncode == 0, "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU"
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", GPU, "· Python", sys.version.split()[0])

from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/krat_avatar"  # weights and inputs survive between Colab sessions
for d in ["entradas", "pesos", "hf", "salidas"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
os.environ["HF_HOME"] = f"{WORK}/hf"
B = "/content/bench"
os.makedirs(B, exist_ok=True)

R = {}      # every measurement, shown in the summary
DITTO = {}  # Ditto configs that ran: name → (data_root, cfg, pasos)

def paso(nombre, fn):
    """Runs one step; a failure is recorded in the summary instead of stopping «Ejecutar todo»."""
    t = time.time()
    try:
        R[nombre] = fn() or {}
        print(f"✔ {nombre} ({time.time() - t:.0f} s)")
    except Exception as e:
        R[nombre] = {"error": str(e)[-600:]}
        print(f"✘ {nombre}: {e}")

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd[:80]}…\n{(r.stderr or r.stdout)[-1500:]}")
    return r.stdout

In [ ]:
# Paso 2 · Entradas (se guardan en Drive: krat_avatar/entradas)
from google.colab import files

ENT = f"{WORK}/entradas"
buscar = lambda nombre: next(iter(sorted(glob.glob(f"{ENT}/{nombre}*"))), None)

if not all(buscar(n) for n in ["foto", "mi_voz", "gemini"]):
    print("Sube foto…, mi_voz… y gemini…. Si cancelas, se usan ejemplos.")
    try:
        for name, data in files.upload().items():
            open(f"{ENT}/{name}", "wb").write(data)
    except Exception as e:
        print("Sin subida:", e)
print({n: buscar(n) for n in ["foto", "mi_voz", "gemini"]})

In [ ]:
# Paso 3 · Instalar. Cada modelo va en su propio Python 3.10 (con uv) con las versiones que piden sus repos,
# así no chocan entre sí ni con el Python de Colab.
sh("pip -q install uv")
sh(f"cd {B} && (test -d ditto || git clone -q --depth 1 https://github.com/antgroup/ditto-talkinghead ditto)")
sh(f"cd {B} && (test -d seed-vc || git clone -q --depth 1 https://github.com/Plachtaa/seed-vc)")
TORCH = "--index-url https://download.pytorch.org/whl/cu121"
DITTO_PY, VOZ_PY = f"{B}/venv_ditto/bin/python", f"{B}/venv_voz/bin/python"

def instalar_ditto():
    sh(f"uv venv -q -p 3.10 {B}/venv_ditto")
    sh(f"uv pip install -q --python {DITTO_PY} torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 {TORCH}")
    sh(f"uv pip install -q --python {DITTO_PY} librosa tqdm filetype imageio imageio-ffmpeg opencv-python-headless "
       "scikit-image cython colored einops mediapipe 'onnxruntime-gpu>=1.21'")
    from huggingface_hub import snapshot_download
    snapshot_download("digital-avatar/ditto-talkinghead", local_dir=f"{WORK}/pesos/ditto",
                      allow_patterns=["ditto_cfg/*", "ditto_pytorch/*", "ditto_onnx/*"])

def instalar_voz():
    sh(f"uv venv -q -p 3.10 {B}/venv_voz")
    sh(f"uv pip install -q --python {VOZ_PY} torch==2.4.0 torchaudio==2.4.0 {TORCH}")
    sh(f"uv pip install -q --python {VOZ_PY} scipy==1.13.1 librosa==0.10.2 huggingface-hub munch==4.0.0 einops==0.8.0 "
       "descript-audio-codec==1.0.0 transformers==4.46.3 soundfile numpy==1.26.4 pyyaml python-dotenv")
    # Seed-VC downloads its weights into ./checkpoints: keep them in Drive.
    os.makedirs(f"{WORK}/pesos/seed-vc", exist_ok=True)
    if not os.path.islink(f"{B}/seed-vc/checkpoints"):
        os.symlink(f"{WORK}/pesos/seed-vc", f"{B}/seed-vc/checkpoints")

def preparar_entradas():
    import cv2
    foto = buscar("foto") or f"{B}/ditto/example/image.png"
    voz = buscar("mi_voz") or f"{B}/seed-vc/examples/reference/s1p1.wav"
    gem = buscar("gemini") or f"{B}/ditto/example/audio.wav"
    img = cv2.imread(foto)
    assert img is not None, f"No pude leer la foto {foto}"
    k = 720 / max(img.shape[:2])  # the live stream will be ≤720 px, so measure at that size
    if k < 1:
        img = cv2.resize(img, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
    cv2.imwrite(f"{B}/foto.png", img)
    sh(f'ffmpeg -loglevel error -y -i "{voz}" -t 30 -ac 1 -ar 22050 {B}/mi_voz.wav')
    sh(f'ffmpeg -loglevel error -y -i "{gem}" -t 15 -ac 1 -ar 16000 {B}/gemini.wav')
    return {"foto": os.path.basename(foto), "mi_voz": os.path.basename(voz), "gemini": os.path.basename(gem),
            "tamaño": f"{img.shape[1]}×{img.shape[0]}"}

paso("Instalar Ditto", instalar_ditto)
paso("Instalar Seed-VC", instalar_voz)
paso("Entradas", preparar_entradas)

In [ ]:
%%writefile /content/bench/bench_ditto.py
"""Ditto throughput on this GPU in online (streaming) mode. Prints nothing useful; writes a JSON."""
import argparse, json, math, os, subprocess, sys, time

import numpy as np

p = argparse.ArgumentParser()
for x in ["--repo", "--data_root", "--cfg", "--foto", "--audio", "--video"]:
    p.add_argument(x, default="")
p.add_argument("--pasos", type=int, default=0)  # diffusion steps; 0 keeps the cfg's default
p.add_argument("json")
a = p.parse_args()
os.chdir(a.repo)
sys.path.insert(0, a.repo)

import torch  # loads the CUDA/cuDNN libs onnxruntime needs
import onnxruntime
getattr(onnxruntime, "preload_dlls", lambda: None)()  # without it ORT can silently fall back to CPU
import cv2, librosa
from stream_pipeline_online import StreamSDK


class Frames:
    """Stands in for Ditto's mp4 writer: timestamps each frame and JPEG-encodes it, as the live stream will."""

    def __init__(self, mp4=None):
        self.t, self.mp4 = [], mp4

    def __call__(self, img, fmt="rgb"):
        cv2.imencode(".jpg", img[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, 80])
        self.t.append(time.perf_counter())
        if self.mp4:
            self.mp4(img, fmt=fmt)

    def close(self):
        if self.mp4:
            self.mp4.close()


sdk = StreamSDK(a.cfg, a.data_root)
kw = {"online_mode": True}
if a.pasos:
    kw["sampling_timesteps"] = a.pasos


def run(audio, video=""):
    t = time.perf_counter()
    sdk.setup(a.foto, video or "/tmp/ditto_descartado.mp4", **kw)
    preparar = time.perf_counter() - t
    frames = Frames(sdk.writer if video else None)
    if not video:
        sdk.writer.close()
    sdk.writer = frames
    sdk.setup_Nd(N_d=math.ceil(len(audio) / 16000 * 25))
    # Same chunking as Ditto's inference.py in online mode: 0.2 s of new audio per chunk.
    cs = (3, 5, 2)
    audio = np.concatenate([np.zeros(cs[0] * 640, np.float32), audio])
    split = int(sum(cs) * 0.04 * 16000) + 80
    t0 = time.perf_counter()
    for i in range(0, len(audio), cs[1] * 640):
        c = audio[i:i + split]
        sdk.run_chunk(np.pad(c, (0, split - len(c))), cs)
    sdk.close()
    n = len(frames.t)
    assert n, "Ditto no generó ningún fotograma"
    return {"fps": round(n / (frames.t[-1] - t0), 1), "primer_fotograma_s": round(frames.t[0] - t0, 2),
            "preparar_avatar_s": round(preparar, 1), "fotogramas": n, "pasos": sdk.sampling_timesteps}


audio, _ = librosa.load(a.audio, sr=16000)
run(audio[:32000])  # warm-up: CUDA kernels and cuDNN autotuning
r = run(audio, a.video)
r["audio_s"] = round(len(audio) / 16000, 1)
if a.video:
    tmp = a.video + ".tmp.mp4"  # where Ditto's writer put the silent video
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", tmp, "-i", a.audio, "-map", "0:v", "-map", "1:a",
                    "-c:v", "copy", "-c:a", "aac", "-shortest", a.video], check=True)
    os.remove(tmp)
json.dump(r, open(a.json, "w"))

In [ ]:
%%writefile /content/bench/bench_voz.py
"""Seed-VC real-time voice conversion on this GPU, block by block as a live stream would. Writes a JSON."""
import argparse, importlib.util, json, os, sys, time, types

import numpy as np

p = argparse.ArgumentParser()
for x in ["--repo", "--ref", "--src", "--out"]:
    p.add_argument(x)
p.add_argument("--pasos", type=int, default=10)
p.add_argument("--ritmo", action="store_true", help="feed blocks at real-time pace, not as fast as possible")
p.add_argument("json")
a = p.parse_args()
os.chdir(a.repo)
sys.path.insert(0, a.repo)

import librosa, soundfile as sf, torch

# real-time-gui.py keeps its GUI under __main__; the model loading and per-block inference are importable.
spec = importlib.util.spec_from_file_location("rt", "real-time-gui.py")
rt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rt)
rt.device = torch.device("cuda")
t = time.perf_counter()
models = rt.load_models(types.SimpleNamespace(fp16=True, checkpoint_path=None, config_path=None))
cargar = time.perf_counter() - t
sr = models[-1]["sampling_rate"]
ref, _ = librosa.load(a.ref, sr=sr)
src, _ = librosa.load(a.src, sr=sr)

# Real-time settings recommended in Seed-VC's README (RTX 3060 Laptop: 150 ms per 180 ms block).
BLOCK, XFADE, CE, DIT, RIGHT = 0.18, 0.04, 2.5, 0.5, 0.02
zc = sr // 50
fr = lambda s: round(s * sr / zc) * zc
block, xfade, extra, right = fr(BLOCK), fr(XFADE), fr(CE), fr(RIGHT)
sola = min(xfade, 4 * zc)
fade = np.sin(0.5 * np.pi * np.linspace(0, 1, sola)) ** 2
win = np.zeros(extra + xfade + zc + block + right, np.float32)
src = np.concatenate([src, np.zeros(block * 2, np.float32)])  # flush the last block
out, ms, prev = [], [], None
t0 = time.perf_counter()
for k, i in enumerate(range(0, len(src) - block + 1, block)):
    if a.ritmo:
        time.sleep(max(0, t0 + k * BLOCK - time.perf_counter()))
    win = np.concatenate([win[block:], src[i:i + block]])
    x16 = torch.from_numpy(librosa.resample(win, orig_sr=sr, target_sr=16000)).to(rt.device)
    torch.cuda.synchronize()
    t = time.perf_counter()
    y = rt.custom_infer(models, ref, a.ref, x16, 320 * block // zc, extra // zc, right // zc,
                        (block + sola + zc) // zc, a.pasos, 0.7, 3.0, CE - DIT)
    torch.cuda.synchronize()
    ms.append((time.perf_counter() - t) * 1000)
    y = y.float().cpu().numpy()
    # shortcut: plain crossfade without the GUI's SOLA offset search; may click faintly between blocks. Fine for judging timbre.
    if prev is not None:
        y[:sola] = y[:sola] * fade + prev * (1 - fade)
    out.append(y[:block])
    prev = y[block:block + sola]

sf.write(a.out, np.concatenate(out), sr)
m = np.array(ms[1:])  # the first block also encodes the reference voice
med = float(np.median(m))
json.dump({"ms_trozo_mediana": round(med), "ms_trozo_p95": round(float(np.percentile(m, 95))),
           "rtf": round(med / 1000 / BLOCK, 2), "latencia_estimada_ms": round((BLOCK + XFADE + RIGHT) * 1000 + med),
           "cargar_s": round(cargar, 1), "pasos": a.pasos}, open(a.json, "w"))

In [ ]:
# Paso 4 · Medir
def medir(*cmds):
    """Runs the bench scripts at the same time, polling total GPU memory. Each script's last arg is its JSON output."""
    logs = [open(c[-1] + ".log", "w") for c in cmds]
    procs = [subprocess.Popen(c, stdout=l, stderr=subprocess.STDOUT) for c, l in zip(cmds, logs)]
    pico = 0
    while any(p.poll() is None for p in procs):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout
        pico = max(pico, int(used.split()[0]))
        time.sleep(0.5)
    res = []
    for c, p, l in zip(cmds, procs, logs):
        l.close()
        if p.returncode:
            raise RuntimeError(open(c[-1] + ".log").read()[-1500:])
        res.append(json.load(open(c[-1])))
    return res, pico

def cmd_ditto(nombre, data_root, cfg, pasos, audio=f"{B}/gemini.wav", video=""):
    return [DITTO_PY, f"{B}/bench_ditto.py", "--repo", f"{B}/ditto", "--data_root", data_root, "--cfg", cfg,
            "--foto", f"{B}/foto.png", "--audio", audio, "--pasos", str(pasos), "--video", video, f"{B}/{nombre}.json"]

def cmd_voz(nombre, ritmo=False):
    return [VOZ_PY, f"{B}/bench_voz.py", "--repo", f"{B}/seed-vc", "--ref", f"{B}/mi_voz.wav", "--src", f"{B}/gemini.wav",
            "--out", f"{B}/gemini_convertido.wav"] + (["--ritmo"] if ritmo else []) + [f"{B}/{nombre}.json"]

def ditto(nombre, data_root, cfg, pasos):
    def fn():
        (r,), pico = medir(cmd_ditto(nombre, data_root, cfg, pasos))
        DITTO[nombre] = (data_root, cfg, pasos)
        return {**r, "vram_mb": pico}
    paso(nombre, fn)

PESOS = f"{WORK}/pesos/ditto"
PT = (f"{PESOS}/ditto_pytorch", f"{PESOS}/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl")
ditto("Ditto PyTorch", *PT, 0)
ditto("Ditto PyTorch · 10 pasos", *PT, 10)

def voz():
    (r,), pico = medir(cmd_voz("voz"))
    return {**r, "vram_mb": pico}
paso("Seed-VC", voz)

from IPython.display import Audio, display
if os.path.exists(f"{B}/gemini_convertido.wav"):
    print("Gemini con tu voz:")
    display(Audio(f"{B}/gemini_convertido.wav"))

In [ ]:
# Paso 5 · TensorRT (opcional). Turing (T4) no tiene motores precompilados: se construyen una vez (de 30 a 40 min) y quedan en Drive.
PROBAR_TRT = True
TRT_DIR = f"{WORK}/pesos/ditto_trt_t4"

def trt():
    sh(f"uv pip install -q --python {DITTO_PY} tensorrt==8.6.1 tensorrt-bindings==8.6.1 tensorrt-libs==8.6.1 "
       "cuda-python==12.6.2.post1 polygraphy --extra-index-url https://pypi.nvidia.com --index-strategy unsafe-best-match")
    libs = sh(f"{DITTO_PY} -c 'import tensorrt_libs, os; print(os.path.dirname(tensorrt_libs.__file__))'").strip()
    # The polygraphy CLI the script shells out to needs TensorRT's .so files on the loader path.
    log = sh(f"cd {B}/ditto && PATH={B}/venv_ditto/bin:$PATH LD_LIBRARY_PATH={libs}:$LD_LIBRARY_PATH "
             f"python scripts/cvt_onnx_to_trt.py --onnx_dir {PESOS}/ditto_onnx --trt_dir {TRT_DIR} 2>&1")
    # The script ignores polygraphy's exit codes, so check that all 12 engines exist.
    n = len(glob.glob(f"{TRT_DIR}/*.engine"))
    assert n >= 12, f"Solo hay {n} de 12 motores TensorRT:\n{log[-1500:]}"

if PROBAR_TRT:
    paso("Convertir a TensorRT", trt)
    if "error" not in R["Convertir a TensorRT"]:
        ditto("Ditto TensorRT", TRT_DIR, f"{PESOS}/ditto_cfg/v0.4_hubert_cfg_trt_online.pkl", 0)

In [ ]:
# Paso 6 · Cara y voz a la vez, como en una llamada: la voz al ritmo real y la cara con la config de Ditto más rápida.
def juntos():
    nombre = max(DITTO, key=lambda n: R[n]["fps"])
    (d, v), pico = medir(cmd_ditto("juntos_ditto", *DITTO[nombre]), cmd_voz("juntos_voz", ritmo=True))
    return {"ditto": nombre, "fps": d["fps"], "primer_fotograma_s": d["primer_fotograma_s"],
            "ms_trozo_mediana": v["ms_trozo_mediana"], "rtf": v["rtf"], "vram_mb": pico}
paso("Juntos", juntos)

# Video to judge quality: the fastest config, with Gemini's audio already in your voice.
VIDEO = f"{WORK}/salidas/ditto_resultado.mp4"
def video():
    nombre = max(DITTO, key=lambda n: R[n]["fps"])
    conv = f"{B}/gemini_convertido.wav"
    medir(cmd_ditto("video", *DITTO[nombre], audio=conv if os.path.exists(conv) else f"{B}/gemini.wav", video=VIDEO))
paso("Vídeo", video)

In [ ]:
# Paso 7 · Resumen
import pandas as pd
from IPython.display import Video, display

df = pd.DataFrame([{"paso": n, **r} for n, r in R.items()]).set_index("paso")
display(df)

j = R.get("Juntos", {})
fps, rtf, vram = j.get("fps", 0), j.get("rtf", 9), j.get("vram_mb", 99999)
if fps >= 20 and rtf < 0.5 and vram < 15000:
    veredicto = "✅ La T4 da: pasar a la Fase 1 con esta arquitectura."
elif fps >= 12:
    veredicto = "🟡 Cerca: bajar la resolución o los pasos de difusión y volver a medir; si no basta, probar MuseTalk."
else:
    veredicto = "❌ No llega: cara con Simli/Anam (de pago) y voz propia en la T4."
print(f"\n{GPU}\nVeredicto: {veredicto}\n")
print("Copia esto y pégaselo a Claude:\n")
print(df.drop(columns="error", errors="ignore").to_markdown())
errores = {n: r["error"] for n, r in R.items() if "error" in r}
if errores:
    print("\nErrores:\n" + json.dumps(errores, ensure_ascii=False, indent=1))
json.dump({"gpu": GPU, "veredicto": veredicto, **R}, open(f"{WORK}/salidas/resultados.json", "w"), ensure_ascii=False, indent=1)
if os.path.exists(VIDEO):
    display(Video(VIDEO, embed=True, width=480))